In [22]:
import math
from typing import Optional, Tuple, Dict


def modinv(a, p):
    """modular inverse of a modulo p (prime)"""
    return pow(a, p - 2, p)


def baby_step_giant_step(alpha, beta, p):
    """
    Solve for x in alpha^x ≡ beta (mod p) with p prime.
    Returns the smallest non-negative solution x < p-1,
    None if no solution exists (should not happen when alpha is a generator).
    """
    n = p - 1  # order of the group Z*_p
    m = math.isqrt(n) + 1  # ceil(sqrt(n))

    table: Dict[int, int] = {}
    cur = 1
    for j in range(m):
        # store only the first occurrence (the smallest j)
        if cur not in table:
            table[cur] = j
        cur = (cur * alpha) % p

    alpha_inv_m = pow(alpha, n - m, p)  # because alpha^{n}=1

    gamma = beta
    for i in range(m):
        if gamma in table:
            j = table[gamma]
            x = i * m + j
            return x % n  # the discrete log (mod p‑1)
        gamma = (gamma * alpha_inv_m) % p

    # No match found. should not happen for a generator
    return None


def test_and_print(alpha, beta, p):
    print(f"\nlog_{alpha} {beta}  (mod {p})")
    if not (1 <= alpha < p and 1 <= beta < p):
        return

    # check that alpha really is a generator, verify that its order is p‑1
    order = p - 1

    # a primitive element must satisfy alpha^{order/q} != 1 for every prime divisor q of order
    def is_primitive(g: int) -> bool:
        tmp = order
        q = 2
        while q * q <= tmp:
            if tmp % q == 0:
                if pow(g, order // q, p) == 1:
                    return False
                while tmp % q == 0:
                    tmp //= q
            q += 1
        if tmp > 1:  # last prime factor
            if pow(g, order // tmp, p) == 1:
                return False
        return True

    if not is_primitive(alpha):
        print("  Warning: the supplied alpha does NOT appear to be primitive.")
        print("  The algorithm will still return a solution, but it may not be unique.")
    else:
        print("  alpha is (very likely) a primitive element modulo p.")

    x = baby_step_giant_step(alpha, beta, p)
    if x is None:
        print("  No discrete logarithm found.")
    else:
        # verify
        assert pow(alpha, x, p) == beta % p
        print(f"  Discrete logarithm:  x = {x}")
        print(f"  Check:  {alpha}^{x} ≡ {beta} (mod {p})")


# 1.  log_106 12375  in Z*_24691
test_and_print(alpha=106, beta=12375, p=24691)

# 2.  log_6 248388  in Z*_458009
test_and_print(alpha=6, beta=248388, p=458009)



log_106 12375  (mod 24691)
  alpha is (very likely) a primitive element modulo p.
  Discrete logarithm:  x = 22392
  Check:  106^22392 ≡ 12375 (mod 24691)

log_6 248388  (mod 458009)
  alpha is (very likely) a primitive element modulo p.
  Discrete logarithm:  x = 232836
  Check:  6^232836 ≡ 248388 (mod 458009)


In [23]:
import sys
import math
from typing import List, Tuple, Dict


def factor_over_base(n, base):
    """
    Return the exponent vector of n with respect to the supplied factor base.
    If n contains a prime outside the base the function raises ValueError.
    """
    factors: Dict[int, int] = {}
    remainder = n
    for p in base:
        cnt = 0
        while remainder % p == 0:
            remainder //= p
            cnt += 1
        if cnt:
            factors[p] = cnt
    if remainder != 1:
        raise ValueError(f"{n} is not B-smooth (remaining factor {remainder})")
    return factors


def vec_from_factors(factors, base):
    """Convert a factor-dictionary into a plain exponent vector ordered as base."""
    return [factors.get(p, 0) for p in base]


def inv_mod(a, m):
    """Multiplicative inverse of a modulo m (a and m must be coprime)."""
    g, x, _ = extended_gcd(a, m)
    if g != 1:
        raise ValueError(f"{a} has no inverse modulo {m}")
    return x % m


def extended_gcd(a, b):
    """Return (g, x, y) such that ax + by = g = gcd(a,b)."""
    if b == 0:
        return (a, 1, 0)
    g, x1, y1 = extended_gcd(b, a % b)
    return (g, y1, x1 - (a // b) * y1)


def solve_linear_system_mod(A, b, mod):
    """
    Solve Ax ≡ b (mod mod) by Gaussian elimination.
    A is a square matrix (len(A) == len(A[0])).
    Returns the unique solution vector x (mod mod).
    """
    n = len(A)
    # make copies we can modify
    M = [row[:] + [b_i] for row, b_i in zip(A, b)]

    # forward elimination
    for col in range(n):
        # find a pivot row with a non‑zero entry in column `col`
        pivot = None
        for row in range(col, n):
            if M[row][col] % mod != 0:
                pivot = row
                break
        if pivot is None:
            raise ValueError("Matrix is singular modulo {}".format(mod))

        # swap pivot row into position col
        M[col], M[pivot] = M[pivot], M[col]

        # scale pivot row so that the leading coefficient becomes 1
        inv = inv_mod(M[col][col] % mod, mod)
        for j in range(col, n + 1):
            M[col][j] = (M[col][j] * inv) % mod

        # eliminate the current column from all other rows
        for row in range(n):
            if row == col:
                continue
            factor = M[row][col] % mod
            if factor == 0:
                continue
            for j in range(col, n + 1):
                M[row][j] = (M[row][j] - factor * M[col][j]) % mod

    # back‑substitution (the matrix is now the identity)
    return [M[i][n] % mod for i in range(n)]


p = 227  # the prime modulus
order = p - 1  # size of the multiplicative group
alpha = 2  # primitive element (generator)
factor_base = [2, 3, 5, 7, 11]


# (a)  Powers of α and their factorizations
exponents = [32, 40, 59, 156]
print("\n(a)  Powers of alpha = 2 and their B-factorisations")
power_vals = []
for e in exponents:
    val = pow(alpha, e, p)
    power_vals.append(val)
    try:
        fac = factor_over_base(val, factor_base)
    except ValueError as err:
        sys.exit(1)
    print(
        f"  alpha^{e} ≡ {val:3d}  =  "
        + " * ".join([f"{prime}^{exp}" for prime, exp in fac.items()])
    )

# (b)  Solve for the logarithms of the factor‑base elements
# unknowns are [log_2 3, log_2 5, log_2 7, log_2 11]
# we already know log_2 2 = 1.
print("\n(b)  Building the linear system")
A: List[List[int]] = []  # coefficient matrix
b: List[int] = []  # right‑hand side (the exponents)

for e, val in zip(exponents, power_vals):
    fac = factor_over_base(val, factor_base)
    vec = vec_from_factors(fac, factor_base)  # [e2, e3, e5, e7, e11]
    # move the contribution of the known log_2 2 to the RHS
    rhs = (e - vec[0] * 1) % order  # subtract 2‑exponent * log_2 2
    # drop the column that corresponds to the prime 2 (its log is known)
    A.append(vec[1:])  # keep only 3,5,7,11
    b.append(rhs)
    print(
        f"  Equation from alpha^{e}:  {e} ≡ "
        f"{vec[0]}*log2 + {vec[1]}*log3 + {vec[2]}*log5 + "
        f"{vec[3]}*log7 + {vec[4]}*log11  (mod {order})"
    )
    print(f"    →  {e} - {vec[0]}·1 ≡ {rhs} (mod {order})")

# Solve the 4×4 system
logs = solve_linear_system_mod(A, b, order)
log2, log3, log5, log7, log11 = 1, *logs  # prepend the known log_2 2 = 1

print("\nDiscrete logarithms of the factor base")
print(f"  log_2 2  = {log2}")
print(f"  log_2 3  = {log3}")
print(f"  log_2 5  = {log5}")
print(f"  log_2 7  = {log7}")
print(f"  log_2 11 = {log11}")

# (c)  Compute log_2 173 using a random multiple (exponent 177)
target = 173
random_exp = 177
random_pow = pow(alpha, random_exp, p)  # 2^177 (mod p)
multiplied = (target * random_pow) % p

print("\n(c)  Random‑multiple step")
print(f"  2^{random_exp} (mod {p}) = {random_pow}")
print(f"  Multiply target:  {target}*{random_pow} ≡ {multiplied} (mod {p})")

# factor the product over the base
fac_c = factor_over_base(multiplied, factor_base)
print("  Factorisation of the product:")
print("   " + " * ".join([f"{prime}^{exp}" for prime, exp in fac_c.items()]))

# Build the equation:
#   log_2(target) + random_exp ≡ sum(e_i * log_2(prime_i))   (mod order)
lhs = (
    sum(
        exp * {2: log2, 3: log3, 5: log5, 7: log7, 11: log11}[prime]
        for prime, exp in fac_c.items()
    )
    - random_exp
) % order

print(f"\n  log_2({target}) ≡ {lhs} (mod {order})")
print(
    f"  Verification: 2^{lhs} (mod {p}) = {pow(alpha, lhs, p)} "
)


(a)  Powers of alpha = 2 and their B-factorisations
  alpha^32 ≡ 176  =  2^4 * 11^1
  alpha^40 ≡ 110  =  2^1 * 5^1 * 11^1
  alpha^59 ≡  60  =  2^2 * 3^1 * 5^1
  alpha^156 ≡  28  =  2^2 * 7^1

(b)  Building the linear system
  Equation from alpha^32:  32 ≡ 4*log2 + 0*log3 + 0*log5 + 0*log7 + 1*log11  (mod 226)
    →  32 - 4·1 ≡ 28 (mod 226)
  Equation from alpha^40:  40 ≡ 1*log2 + 0*log3 + 1*log5 + 0*log7 + 1*log11  (mod 226)
    →  40 - 1·1 ≡ 39 (mod 226)
  Equation from alpha^59:  59 ≡ 2*log2 + 1*log3 + 1*log5 + 0*log7 + 0*log11  (mod 226)
    →  59 - 2·1 ≡ 57 (mod 226)
  Equation from alpha^156:  156 ≡ 2*log2 + 0*log3 + 0*log5 + 1*log7 + 0*log11  (mod 226)
    →  156 - 2·1 ≡ 154 (mod 226)

Discrete logarithms of the factor base
  log_2 2  = 1
  log_2 3  = 46
  log_2 5  = 11
  log_2 7  = 154
  log_2 11 = 28

(c)  Random‑multiple step
  2^177 (mod 227) = 123
  Multiply target:  173*123 ≡ 168 (mod 227)
  Factorisation of the product:
   2^3 * 3^1 * 7^1

  log_2(173) ≡ 26 (mod 226)
  Ve

In [24]:
import sys
import math
from pathlib import Path
from typing import List, Tuple, Dict

P = 649_663_831                # prime modulus
ALPHA = 295_850                # generator
BETA = 56_181_604              # α^a (mod P)

def int_to_word(m):
    letters = []
    for _ in range(3):
        letters.append(chr(ord('A') + (m % 26)))
        m //= 26
    return ''.join(reversed(letters))

# Recover the secret exponent a  (Beta = alpha^a mod P)
print("Recovering the secret key a = log_alpha Beta")
a = baby_step_giant_step(ALPHA, BETA, P)
print(f"  a = {a}")


# Read the ciphertext file
cipher_path = Path("cipher.txt")
ciphertext: List[Tuple[int, int]] = []
with cipher_path.open("r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            c1_str, c2_str = line.split(",")
            c1 = int(c1_str)
            c2 = int(c2_str)
            ciphertext.append((c1, c2))
        except ValueError:
            sys.exit()


# Decrypt each block
ONE_BASED = False          # set to True if the source used A=1,…,Z=26
plain_blocks: List[str] = []

for idx, (c1, c2) in enumerate(ciphertext, start=1):
    # shared secret s = c1^a (mod P)
    s = pow(c1, a, P)
    s_inv = modinv(s, P)

    # plaintext integer m = c2 * s^{-1} (mod P)
    m = (c2 * s_inv) % P

    # translate to three letters
    try:
        block = int_to_word(m)
    except ValueError as e:
        sys.exit()

    plain_blocks.append(block)

# Display the quote
plain_text = "".join(plain_blocks)

print("\nDecrypted quotes")
print(plain_text)

Recovering the secret key a = log_alpha Beta
  a = 379053

Decrypted quotes
IAMAVERYSTRONGBELIEVERINLISTENINGANDLEARNINGFROMOTHERSRBGINSBURGWECANNOTSOLVEPROBLEMSWITHTHEKINDOFTHINKINGWEEMPLOYEDWHENWECAMEUPWITHTHEMEINSTEINLEARNASIFYOUWILLLIVEFOREVERLIVELIKEYOUWILLDIETOMORROWGANDHITHEPESSIMISTSEESDIFFICULTYINEVERYOPPORTUNITYTHEOPTIMISTSEESOPPORTUNITYINEVERYDIFFICULTYCHURCHILLITSKINDOFFUNTODOTHEIMPOSSIBLEDISNEYYOUHAVEBRAINSINYOURHEADYOUHAVEFEETINYOURSHOESYOUCANSTEERYOURSELFANYDIRECTIONYOUCHOOSEYOUREONYOUROWNANDYOUKNOWWHATYOUKNOWANDYOUARETHEGUYWHOLLDECIDEWHERETOGODRSEUSSZZ
